## Framework Overview  

This framework is designed to streamline the **data preprocessing pipeline** by implementing multiple algorithms for:  

- **Data Cleansing**  
- **Outlier Detection & Handling**  
- **Feature Selection**  
- **Model Selection**  

At each step, we evaluate the effectiveness of different algorithms using a **shallow Decision Tree model** and  a **KNN Model**. This allows us to determine the best preprocessing strategy based on performance.  


## Importing Packages

In [79]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split #Weak Model Test
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score #Weak Model Test
from sklearn.tree import DecisionTreeClassifier #Weak Model Test
from sklearn.neighbors import KNeighborsClassifier #Weak Model Test
from sklearn.impute import SimpleImputer, KNNImputer #Handle Missing
from sklearn.linear_model import LinearRegression #Handle Missing
from sklearn.experimental import enable_iterative_imputer #Handle Missing(Imported Because of API Change)
from sklearn.impute import IterativeImputer #Handle Missing
from sklearn.neighbors import LocalOutlierFactor #Tame Outlier
from sklearn.ensemble import IsolationForest #Tame Outlier
from scipy.stats import zscore #Tame Outlier

## Convert into Numeric Values

In [45]:
def ConvertToNumeric(data):
    cols = data.columns
    num_cols = data._get_numeric_data().columns
    categorical_columns = list(set(cols) - set(num_cols))  
    
    for column in categorical_columns:
        categories = list(data[column].astype(str).unique())  
        data[column] = data[column].map(lambda x: categories.index(str(x))) 
    
    return data

## Loading Data

In [46]:
def load_data(path, y_column):
    data=pd.read_excel(path)
    data=ConvertToNumeric(data)
    Y=data[y_column]
    X=data.drop(columns=y_column)
    return X,Y

## Implementing Test Models

In [83]:
def train_and_evaluate(X, y):
    """Trains weak models (KNN and Decision Tree) and evaluates them with multiple metrics."""
    
    print(len(X),len(y),X.isna().sum().sum(),y.isna().sum().sum())
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    knn_model = KNeighborsClassifier(n_neighbors=3)
    knn_model.fit(X_train, y_train)
    knn_pred = knn_model.predict(X_test)
    
    knn_accuracy = accuracy_score(y_test, knn_pred)
    knn_precision = precision_score(y_test, knn_pred, average='weighted', zero_division=1)
    knn_recall = recall_score(y_test, knn_pred, average='weighted', zero_division=1)
    knn_f1 = f1_score(y_test, knn_pred, average='weighted')

    dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_model.fit(X_train, y_train)
    dt_pred = dt_model.predict(X_test)
    
    dt_accuracy = accuracy_score(y_test, dt_pred)
    dt_precision = precision_score(y_test, dt_pred, average='weighted', zero_division=1)
    dt_recall = recall_score(y_test, dt_pred, average='weighted', zero_division=1)
    dt_f1 = f1_score(y_test, dt_pred, average='weighted')

    # print("KNN (k=3) Metrics:")
    # print(f"  Accuracy: {knn_accuracy:.4f}, Precision: {knn_precision:.4f}, Recall: {knn_recall:.4f}, F1-score: {knn_f1:.4f}")
    
    # print("\nDecision Tree (max_depth=3) Metrics:")
    # print(f"  Accuracy: {dt_accuracy:.4f}, Precision: {dt_precision:.4f}, Recall: {dt_recall:.4f}, F1-score: {dt_f1:.4f}")

    weighted_score = (knn_accuracy + knn_precision + knn_recall + knn_f1) + 2 * (dt_accuracy + dt_precision + dt_recall + dt_f1) 

    return weighted_score


# Handling Missing Data

These functions help clean our dataset by handling missing values efficiently and selecting the best imputation method.

## Available Algorithms  
- **Dropping Methods:**  
  - Drop columns with null values exceeding a threshold  
  - Drop rows with missing values  

- **Statistical Imputation:**  
  - Impute with **mean** or **median**  
  - Impute with **class-specific mean** or **median**  

- **Forward & Backward Filling:**  
  - **Forward fill (ffill)**  
  - **Backward fill (bfill)**  
  - **Interpolate** missing values  

- **Model-Based Imputation:**  
  - **Model Imputation** (predict missing values using a simple model)  
  - **Iterative Model Imputation** (refines predictions iteratively)  
  - **KNN Imputation** (fills missing values based on k-nearest neighbors)  


In [63]:
def drop_rows(X, Y):
    """Drops rows with any missing values."""
    mask=X.notna().all(axis=1)
    return X[mask], Y[mask]

def impute_mean(X, Y):
    """Fills missing values with column mean while leaving non-missing values intact."""
    
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)
    
    X_filled = X.copy()
    X_filled[X.isnull()] = X_imputed_df[X.isnull()]
    
    return X_filled,Y

def impute_median(X, Y):
    """Fills missing values with column median while leaving non-missing values intact."""
    
    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)
    X_filled = X.copy()
    X_filled[X.isnull()] = X_imputed_df[X.isnull()]
    
    return X_filled,Y

def impute_class_mean(X, Y):
    """Fills missing values with the mean of each class in Y, ensuring no NaNs remain."""
    X_copy = X.copy()
    for col in X.columns:
        grouped_means = X.groupby(Y)[col].transform(lambda x: x.mean())
        X_copy[col] = X[col].fillna(grouped_means)  # Fill NaNs with class mean
        
        # If NaNs still exist (e.g., entire class missing), fill with global mean
        if X_copy[col].isna().sum() > 0:
            X_copy[col].fillna(X[col].mean(), inplace=True)
    
    return X_copy, Y

def impute_class_median(X, Y):
    """Fills missing values with the median of each class in Y."""
    X_copy = X.copy()
    for col in X.columns:
        X_copy[col] = X.groupby(Y)[col].transform(lambda x: x.fillna(x.median()))

        if X_copy[col].isna().sum() > 0:
            X_copy[col].fillna(X[col].mean(), inplace=True)
    return X_copy,Y

def ffill(X, Y):
    """Fills missing values with the previous row (forward fill) and ensures no NaNs remain."""
    X_filled = X.ffill() 
    X_filled = X_filled.bfill()  
    
    return X_filled, Y

def bfill(X, Y):
    """Fills missing values with the next row (backward fill)."""
    X_filled = X.bfill()  
    X_filled = X_filled.ffill() 
    return X_filled,Y

def interpolate(X, Y):
    """Interpolates missing values linearly and ensures no NaNs remain."""
    X_filled = X.interpolate(method="linear", limit_direction="both")
    return X_filled, Y



def Model_imputation(X, Y):
    """Uses a simple regression model to impute missing values."""
    
    X_copy = X.copy()
    
    for col in X.columns:
        missing_mask = X_copy[col].isna()
        if missing_mask.sum() > 0:
            
            known_data = X_copy.dropna()  # Ensure all rows are clean for model training
            known_X = known_data.drop(columns=[col])
            known_y = known_data[col]
            
            missing_X = X_copy.loc[missing_mask].drop(columns=[col])
            
            # If missing_X still contains NaNs, fill them with column mean before prediction
            if missing_X.isna().sum().sum() > 0:
                missing_X = missing_X.fillna(known_X.mean())

            # Check if we have enough data to train the model
            if len(known_X) > 0 and len(missing_X) > 0:
                model = LinearRegression()
                model.fit(known_X, known_y)
                X_copy.loc[missing_mask, col] = model.predict(missing_X)

    return X_copy, Y

def Iterative_model_Imputation(X, Y):
    """Uses iterative imputation (sklearn's IterativeImputer) to fill only null values."""
    
    imputer = IterativeImputer()
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)
    
    X_filled = X.copy()
    X_filled[X.isnull()] = X_imputed_df[X.isnull()]
    
    return X_filled,Y

def KNN_Imputation(X, Y, n_neighbors=5):
    """Uses KNN to fill missing values while leaving non-missing values intact."""
    
    imputer = KNNImputer(n_neighbors=n_neighbors)
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)
    
    X_filled = X.copy()
    X_filled[X.isnull()] = X_imputed_df[X.isnull()]
    
    return X_filled,Y

algorithm_functions_clean_data = {
    "drop_rows": drop_rows,
    "impute_mean": impute_mean,
    "impute_median": impute_median,
    "impute_class_mean": impute_class_mean,
    "impute_class_median": impute_class_median,
    "ffill": ffill,
    "bfill": bfill,
    "interpolate": interpolate,
    "Model_imputation": Model_imputation,
    "Iterative_model_Imputation": Iterative_model_Imputation,
    "KNN_Imputation": KNN_Imputation,
}


In [49]:
def drop_columns(X,threshold=0.3):
    return X.dropna(axis=1, thresh=(len(X)*threshold), inplace=False)

def handling_missing_data(X, Y):
    """Applies different missing data handling algorithms, selects the best one, and returns the transformed dataset."""
    X = drop_columns(X) 
    Y = Y.dropna(inplace=False)
    X = X.loc[Y.index]
    x_copy = X.copy()
    y_copy = Y.copy()
    acc_holder = {}

    best_algo = None
    best_accuracy = -1

    for name, func in algorithm_functions_clean_data.items():
        X_transformed,Y_transformed = func(x_copy.copy(), y_copy.copy())
        print("*"*10,name,"*"*10)

        accuracy = train_and_evaluate(X_transformed, Y_transformed)
        acc_holder[name] = accuracy
        print("*"*10,accuracy,"*"*10)

        if accuracy > best_accuracy:  
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_clean_data[best_algo](x_copy.copy(), y_copy.copy())
        
    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

# Taming Outliers

These functions help clean our dataset from outliers efficiently and selecting the best outlier detection algorithm.

## Available Algorithms  
- **IQR Method(1.5):**  
  - Take 1.5 times the IQR and then subtract this value from Q1 and add this value to Q3


- **LOF:**  
  - For any data object **q**, the **LOF score** is computed as the ratio of the **average local density** of its **k-nearest neighbors** to its **own local density** **[25]**.  

  $$
  LOF(q) = \frac{\sum_{x \in N_k(q)} lrd(x)}{|N_k(q)| \times lrd(q)}
  $$

  where the **local reachability density (lrd)** of **q** is given by:  

  $$
  lrd(q) = \frac{|N_k(q)|}{\sum_{x \in N_k(q)} \max(\text{dist}_k(x, D), \text{dist}(q, x))}
  $$

- **SP:**  
   - Employ a **scoring measure** based on the nearest neighbor (**k = 1**) within random sub-samples (**S ⊂ D**).  

    $$  S_p(q) = \min_{{x \in S}} \text{dist}(q, x) $$

    where **dist(q, x)** represents the distance between **q** and **x**. 

- **iForest:**  
  - A **random split** is performed on a randomly selected feature.  
  - The partitioning continues until either:  
    - Each node contains only **one data object**, or  
    - The tree reaches its **height limit**. 

  $$
  iForest(q) = \frac{1}{t} \sum_{i=1}^{t} l_i(q)
  $$
 

- **iNNe:**  
  - This method builds **hyperspheres** using all dimensions of the dataset. The **isolation score** of a data object **q** is defined as:  

  $$
  I(q) =
  \begin{cases} 
  \tau (\eta_{cnn}(q)), & \text{if } q \in \bigcup_{c \in S} B(c) \\  
  1 - \tau (cnn(q)), & \text{otherwise}  
  \end{cases}
  $$


In [88]:
def IQR(X, Y):
    """Adjust outliers based on IQR min-max whiskers."""
    Q1 = X.quantile(0.25)
    Q3 = X.quantile(0.75)
    IQR = Q3 - Q1
    min_whisker = Q1 - 1.5 * IQR
    max_whisker = Q3 + 1.5 * IQR
    
    
    X_adjusted = pd.DataFrame(X.clip(lower=min_whisker, upper=max_whisker, axis=1))
    return X_adjusted, Y

def LOF(X, Y):
    """Adjust based on LOF anomaly score."""
    clf = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
    anomaly_scores = clf.fit_predict(X)
    X_scores = clf.negative_outlier_factor_
    
    adjustment_factor = np.abs(X_scores) / np.max(np.abs(X_scores))
    
    X_adjusted = X * (1 - adjustment_factor[:, np.newaxis])
    return X_adjusted, Y

def IsolationForestOutlier(X, Y):
    """Adjust outliers based on Isolation Forest isolation score."""
    iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
    iso.fit(X)
    
    scores = iso.decision_function(X)
    
    adjustment_factor = (scores - scores.min()) / (scores.max() - scores.min())
    X_adjusted = X * (1 - adjustment_factor[:, np.newaxis])
    
    return X_adjusted, Y


def SP(X, Y):
    """Adjusts values based on Standardization Projection (capping beyond 3 std dev)."""
    
    # Compute Z-scores while ignoring NaNs
    Z_scores = np.abs(zscore(X, nan_policy='omit'))  
    
    cap_value = 3
    Z_scores = np.clip(Z_scores, None, cap_value)

    # Handle NaNs before division to prevent propagation
    Z_scores = np.nan_to_num(Z_scores, nan=0)  # Replace NaNs with 0
    
    # Prevent division by zero
    max_Z = np.max(Z_scores, axis=0)
    max_Z[max_Z == 0] = 1  # Replace zero max values with 1 to avoid division error
    
    # Apply transformation for each feature separately (column-wise)
    X_adjusted = X.copy()
    
    for col in X.columns:
        # Get the mask for non-NaN values in each column
        mask = ~X_adjusted[col].isna()
        
        # Apply transformation to non-NaN values
        X_adjusted[col][mask] = X_adjusted[col][mask] * (Z_scores[mask, X.columns.get_loc(col)] / max_Z[X.columns.get_loc(col)])
    
    return X_adjusted, Y

def IsolationNNe(X, Y):
    """Adjust outliers based on Isolation Forest distance score."""
    iso = IsolationForest(contamination=0.05, n_estimators=200, random_state=42)
    iso.fit(X)
    
    dist = iso.decision_function(X)
    
    adjustment_factor = np.abs(dist) / np.max(np.abs(dist))
    X_adjusted = X * (1 - adjustment_factor[:, np.newaxis])
    
    return X_adjusted, Y


algorithm_functions_tame_outlier = {
    "IQR": IQR,
    "LOF": LOF,
    "iForest": IsolationForestOutlier,
    "SP": SP,
    "iNNe": IsolationNNe,
}


In [81]:
def taming_outliers(X,Y):
    acc_holder = {}
    x_copy = X.copy()
    y_copy = Y.copy()
    best_algo = None
    best_accuracy = 0


    for name, func in algorithm_functions_tame_outlier.items():
        X_Adjusted,Y_Adjusted = func(x_copy.copy(), y_copy.copy())
        print(name)
        accuracy = train_and_evaluate(X_Adjusted, Y_Adjusted)
        acc_holder[name] = accuracy
        print(accuracy)


        if accuracy > best_accuracy:  
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_tame_outlier[best_algo](x_copy.copy(), y_copy.copy())
        
    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

In [52]:
# X,Y=load_data(path='./Shariati_final_results.xlsx',y_column='کووید 19-RT-PCR(COVID-19)')

In [77]:
x_copy_clean_data, y_copy_clean_data, best_algo_clean_data, best_accuracy_clean_data,acc_holder_clean_data=handling_missing_data(X,Y)

********** drop_rows **********
2699 2699
********** 12.0 **********
********** impute_mean **********
10933 10933
********** 11.95312978015674 **********
********** impute_median **********
10933 10933
********** 11.95312978015674 **********


/tmp/ipykernel_10454/3182308454.py:38: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_copy[col].fillna(X[col].mean(), inplace=True)


********** impute_class_mean **********
10933 10933
********** 11.94949817055797 **********


/tmp/ipykernel_10454/3182308454.py:49: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_copy[col].fillna(X[col].mean(), inplace=True)


********** impute_class_median **********
10933 10933
********** 11.95312978015674 **********
********** ffill **********
10933 10933
********** 11.948954016906367 **********
********** bfill **********
10933 10933
********** 11.95041661823553 **********
********** interpolate **********
10933 10933
********** 11.951830661661338 **********
********** Model_imputation **********
10933 10933


KeyboardInterrupt: 

In [89]:
x_copy_tame_outlier, y_copy_tame_outlier, best_algo_tame_outlier, best_accuracy_tame_outlier,acc_holder_tame_outlier=taming_outliers(x_copy_clean_data,y_copy_clean_data)


IQR
2699 2699 0 0
12.0
LOF
2699 2699 0 0
12.0
iForest
2699 2699 0 0
12.0


/tmp/ipykernel_10454/481352470.py:61: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  X_adjusted[col][mask] = X_adjusted[col][mask] * (Z_scores[mask, X.columns.get_loc(col)] / max_Z[X.columns.get_loc(col)])
/tmp/ipykernel_10454/481352470.py:61

SP
2699 2699 0 0
11.990739024473964
iNNe
2699 2699 0 0
12.0
